# Lab 5: ablation улучшений AI

Ноутбук воспроизводит ablation на 42 frozen review-кейсах. Каждый механизм отключается отдельно; результаты сохраняются в `artifacts/experiments/lab5-ablation-results.json`.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
completed = subprocess.run(
    [sys.executable, '-m', 'ml.experiments.run_ablation'],
    cwd=repo, capture_output=True, text=True
)
completed.check_returncode()
report = json.loads((repo / 'artifacts/experiments/lab5-ablation-results.json').read_text(encoding='utf-8'))
print(f"Ablation regenerated: {report['review_case_count']} review cases")
print(f"Full semantic rules agreement: {report['ablation']['full_semantic_rules']['agreement']:.2%}")

Ablation regenerated: 42 review cases
Full semantic rules agreement: 92.86%


In [2]:
print(f"{'variant':30} {'agreement':>10} {'delta vs full':>15}")
for name, metrics in report['ablation'].items():
    print(f"{name:30} {metrics['agreement']:>9.2%} {metrics['delta_vs_full_pp']:>13.2f} pp")

variant                         agreement   delta vs full
full_semantic_rules               92.86%          0.00 pp
without_phrase_aliases            88.10%         -4.76 pp
without_stemming                  90.48%         -2.38 pp
without_source_precision          90.48%         -2.38 pp
without_contradiction_guard       88.10%         -4.76 pp
raw_lexical_overlap               71.43%        -21.43 pp


## Решение

Оставляем полный набор semantic rules. Наибольший отдельный вклад дают phrase aliases и contradiction guard: отключение каждого снижает agreement на 4,76 п.п. Сырой lexical overlap хуже полного варианта на 21,43 п.п. Известное ограничение: три расхождения связаны с эталонами, где почти дословный ответ размечен как `partially_correct`.